
# V1 Stage 1 — Part 23 Visualization Notebook

This notebook is the **single end-to-end visualization workflow** for Stage 1.

It reads the validated Stage 1 research outputs and creates:

1. Price + average touch examples
2. Forward-return distributions
3. MFE / MAE distributions
4. Average / timeframe / asset / touch-zone comparisons
5. Market-regime comparisons
6. Profit-target reach charts
7. Target-vs-stop first-hit charts
8. Trailing / post-activation charts
9. Final visualization index and validation

Primary research candidate used for event-level charts:

- Asset: `SOLUSDT`
- Timeframe: `5m`
- Average: `15d`
- Touch zone: `±0.5%`

All figures are saved under `results/figures/`.


## 0. Imports and configuration

In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -------------------------------------------------------------------
# Project configuration
# -------------------------------------------------------------------

ASSET = "SOLUSDT"
TIMEFRAME = "5m"
AVERAGE_TYPE = "15d"
TOUCH_ZONE = 0.5

TABLES_DIR = Path("results/tables")
REPORTS_DIR = Path("results/reports")
FIGURE_ROOT = Path("results/figures")
RESEARCH_DIR = Path("data/research")
PROCESSED_DIR = Path("data/processed")

FIGURE_DIRS = {
    "price_average": FIGURE_ROOT / "01_price_average",
    "forward_returns": FIGURE_ROOT / "02_forward_returns",
    "mfe_mae": FIGURE_ROOT / "03_mfe_mae",
    "average": FIGURE_ROOT / "04_average_comparison",
    "timeframe": FIGURE_ROOT / "05_timeframe_comparison",
    "asset": FIGURE_ROOT / "06_asset_comparison",
    "zone": FIGURE_ROOT / "07_touch_zone_comparison",
    "regime": FIGURE_ROOT / "08_regime_comparison",
    "target_first_hit": FIGURE_ROOT / "09_target_first_hit",
    "trailing": FIGURE_ROOT / "10_trailing",
}

for directory in FIGURE_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

HORIZONS = ["5m", "15m", "30m", "1h", "4h", "24h"]

print("Visualization directories ready.")


## 1. Helper functions

In [ ]:

def find_existing_file(directory, exact_names=(), contains=(), suffix=".csv"):
    directory = Path(directory)

    for name in exact_names:
        path = directory / name
        if path.exists():
            return path

    if not directory.exists():
        return None

    candidates = []
    for path in directory.rglob(f"*{suffix}"):
        lower = path.name.lower()
        if all(term.lower() in lower for term in contains):
            candidates.append(path)

    if not candidates:
        return None

    return sorted(candidates, key=lambda p: (len(str(p)), str(p)))[0]


def load_required_csv(directory, exact_names=(), contains=()):
    path = find_existing_file(
        directory=directory,
        exact_names=exact_names,
        contains=contains,
        suffix=".csv",
    )

    if path is None:
        raise FileNotFoundError(
            f"Could not find required CSV. exact_names={exact_names}, contains={contains}"
        )

    print("Loaded:", path)
    return pd.read_csv(path)


def first_existing_column(df, candidates, required=True):
    for column in candidates:
        if column in df.columns:
            return column

    if required:
        raise KeyError(
            "None of these columns were found: "
            + ", ".join(candidates)
            + "\nAvailable columns:\n"
            + "\n".join(df.columns)
        )

    return None


def normalize_setup_columns(df):
    df = df.copy()

    if "average_type" in df.columns:
        df["average_type"] = (
            df["average_type"]
            .astype(str)
            .str.lower()
            .str.strip()
        )

    if "touch_zone" in df.columns:
        df["touch_zone"] = pd.to_numeric(
            df["touch_zone"],
            errors="coerce",
        )

    return df


def filter_primary_setup(df):
    df = normalize_setup_columns(df)

    mask = pd.Series(True, index=df.index)

    if "asset" in df.columns:
        mask &= df["asset"].eq(ASSET)

    if "timeframe" in df.columns:
        mask &= df["timeframe"].eq(TIMEFRAME)

    if "average_type" in df.columns:
        mask &= df["average_type"].eq(AVERAGE_TYPE)

    if "touch_zone" in df.columns:
        mask &= np.isclose(
            df["touch_zone"].astype(float),
            TOUCH_ZONE,
        )

    return df.loc[mask].copy()


def save_figure(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    fig.savefig(
        path,
        dpi=160,
        bbox_inches="tight",
    )
    plt.close(fig)

    return path


def add_value_labels(ax, bars, fmt="{:.3f}"):
    for bar in bars:
        value = bar.get_height()

        if value >= 0:
            va = "bottom"
        else:
            va = "top"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value,
            fmt.format(value),
            ha="center",
            va=va,
            fontsize=9,
        )


def save_bar_chart(
    labels,
    values,
    title,
    ylabel,
    output_file,
    reference_line=None,
    fmt="{:.3f}",
):
    fig, ax = plt.subplots(figsize=(10, 6))

    positions = np.arange(len(labels))
    bars = ax.bar(positions, values)

    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)

    if reference_line is not None:
        ax.axhline(
            reference_line,
            linestyle="--",
            linewidth=1.2,
            label=f"Reference = {reference_line}",
        )
        ax.legend()

    add_value_labels(ax, bars, fmt=fmt)
    fig.tight_layout()

    return save_figure(fig, output_file)


def discover_primary_event_file():
    preferred = (
        RESEARCH_DIR
        / ASSET
        / f"{ASSET}_{TIMEFRAME}_touch_events_with_trailing_research.parquet"
    )

    if preferred.exists():
        return preferred

    required = {
        "average_type",
        "touch_zone",
        "return_24h_pct",
    }

    candidates = []

    for path in (RESEARCH_DIR / ASSET).rglob("*.parquet"):
        try:
            columns = set(pd.read_parquet(path).columns)
        except Exception:
            continue

        if required.issubset(columns):
            candidates.append(path)

    if not candidates:
        raise FileNotFoundError(
            "Could not find the primary event-level research parquet."
        )

    return sorted(candidates, key=lambda p: len(str(p)))[0]


def discover_price_average_file():
    required = {
        "open_time",
        "close",
        "average_7d",
        "average_15d",
        "average_30d",
    }

    candidates = []

    for root in [PROCESSED_DIR, RESEARCH_DIR]:
        if not root.exists():
            continue

        for path in root.rglob("*.parquet"):
            lower = path.name.lower()

            if ASSET.lower() not in lower:
                continue

            if TIMEFRAME.lower() not in lower:
                continue

            try:
                columns = set(pd.read_parquet(path).columns)
            except Exception:
                continue

            if required.issubset(columns):
                candidates.append(path)

    if not candidates:
        raise FileNotFoundError(
            "Could not find SOLUSDT 5m price data with 7D/15D/30D averages."
        )

    return sorted(
        candidates,
        key=lambda p: ("research" in str(p).lower(), len(str(p))),
    )[0]


def attach_regime(price):
    price = price.copy()

    complete = price[
        ["average_7d", "average_15d", "average_30d"]
    ].notna().all(axis=1)

    bullish = (
        complete
        & (price["average_7d"] > price["average_15d"])
        & (price["average_15d"] > price["average_30d"])
    )

    bearish = (
        complete
        & (price["average_7d"] < price["average_15d"])
        & (price["average_15d"] < price["average_30d"])
    )

    price["market_regime"] = "neutral"
    price.loc[bullish, "market_regime"] = "bullish"
    price.loc[bearish, "market_regime"] = "bearish"
    price.loc[~complete, "market_regime"] = pd.NA

    return price


def find_mae_column(columns):
    return first_existing_column(
        pd.DataFrame(columns=list(columns)),
        [],
    )


In [ ]:

# Dedicated MAE/MFE candidate lists used later.

MAE_CANDIDATES = [
    "mae_24h_pct",
    "max_downside_24h_pct",
    "downside_24h_pct",
]

MFE_CANDIDATES = [
    "mfe_24h_pct",
    "max_upside_24h_pct",
    "upside_24h_pct",
]


def resolve_from_columns(columns, candidates, label):
    for candidate in candidates:
        if candidate in columns:
            return candidate

    raise KeyError(
        f"Could not find {label}. Tried: {candidates}\n"
        f"Available columns:\n" + "\n".join(sorted(columns))
    )


## 2. Load validated Stage 1 tables and event-level data

In [ ]:

summary = load_required_csv(
    TABLES_DIR,
    exact_names=("stage1_configuration_summary.csv",),
)

average_summary = load_required_csv(
    TABLES_DIR,
    exact_names=("average_overall_summary.csv",),
    contains=("average", "overall", "summary"),
)

timeframe_summary = load_required_csv(
    TABLES_DIR,
    exact_names=("timeframe_comparison_overall.csv",),
    contains=("timeframe", "comparison", "overall"),
)

asset_summary = load_required_csv(
    TABLES_DIR,
    exact_names=("asset_comparison_overall.csv",),
    contains=("asset", "comparison", "overall"),
)

regime_primary = load_required_csv(
    TABLES_DIR,
    exact_names=("regime_primary_setup_comparison.csv",),
    contains=("regime", "primary", "setup"),
)

profit_reach = load_required_csv(
    TABLES_DIR,
    exact_names=("profit_reach_statistics.csv",),
    contains=("profit", "reach", "statistics"),
)

first_hit = load_required_csv(
    TABLES_DIR,
    exact_names=("first_hit_statistics.csv",),
    contains=("first", "hit", "statistics"),
)

trailing = load_required_csv(
    TABLES_DIR,
    exact_names=("trailing_outcome_comparison.csv",),
    contains=("trailing", "outcome", "comparison"),
)

event_file = discover_primary_event_file()
price_file = discover_price_average_file()

print("\nEvent-level parquet:", event_file)
print("Price/average parquet:", price_file)

event_columns = pd.read_parquet(event_file).columns.tolist()

needed_event_columns = [
    "asset",
    "timeframe",
    "average_type",
    "touch_zone",
    "timestamp",
    *[f"return_{h}_pct" for h in HORIZONS],
]

mae_column = resolve_from_columns(
    set(event_columns),
    MAE_CANDIDATES,
    "24h MAE column",
)

mfe_column = resolve_from_columns(
    set(event_columns),
    MFE_CANDIDATES,
    "24h MFE column",
)

for column in [mae_column, mfe_column]:
    if column not in needed_event_columns:
        needed_event_columns.append(column)

available_event_columns = [
    column
    for column in needed_event_columns
    if column in event_columns
]

events = pd.read_parquet(
    event_file,
    columns=available_event_columns,
)

events = filter_primary_setup(events)

if "timestamp" in events.columns:
    events["timestamp"] = pd.to_datetime(
        events["timestamp"],
        utc=True,
    )

events["mae_24h_pct"] = pd.to_numeric(
    events[mae_column],
    errors="coerce",
)

events["mfe_24h_pct"] = pd.to_numeric(
    events[mfe_column],
    errors="coerce",
)

print("\nPrimary event rows:", len(events))
print("MAE column:", mae_column)
print("MFE column:", mfe_column)


## 3. Price + average touch examples

In [ ]:

price = pd.read_parquet(
    price_file,
    columns=[
        "open_time",
        "close",
        "average_7d",
        "average_15d",
        "average_30d",
    ],
)

price["open_time"] = pd.to_datetime(
    price["open_time"],
    utc=True,
)

price = (
    price
    .sort_values("open_time")
    .drop_duplicates(subset=["open_time"])
    .reset_index(drop=True)
)

price = attach_regime(price)

touch_events = events[
    ["timestamp"]
].drop_duplicates().copy()

touch_events = touch_events.merge(
    price[
        ["open_time", "market_regime"]
    ],
    left_on="timestamp",
    right_on="open_time",
    how="left",
    validate="many_to_one",
)

price_average_outputs = []

for regime in ["neutral", "bearish", "bullish"]:
    candidates = touch_events[
        touch_events["market_regime"].eq(regime)
    ].copy()

    if candidates.empty:
        print(f"Skipping {regime}: no matching event")
        continue

    # Middle event avoids cherry-picking.
    event_time = candidates.iloc[len(candidates) // 2]["timestamp"]

    start = event_time - pd.Timedelta(hours=12)
    end = event_time + pd.Timedelta(hours=24)

    window = price[
        price["open_time"].between(start, end)
    ].copy()

    window["zone_upper"] = (
        window["average_15d"]
        * (1 + TOUCH_ZONE / 100)
    )

    window["zone_lower"] = (
        window["average_15d"]
        * (1 - TOUCH_ZONE / 100)
    )

    fig, ax = plt.subplots(figsize=(15, 8))

    ax.plot(
        window["open_time"],
        window["close"],
        label="5m close",
        linewidth=1.2,
    )

    ax.plot(
        window["open_time"],
        window["average_7d"],
        label="7D average",
        linewidth=1.0,
    )

    ax.plot(
        window["open_time"],
        window["average_15d"],
        label="15D average",
        linewidth=1.5,
    )

    ax.plot(
        window["open_time"],
        window["average_30d"],
        label="30D average",
        linewidth=1.0,
    )

    ax.fill_between(
        window["open_time"],
        window["zone_lower"],
        window["zone_upper"],
        alpha=0.15,
        label="15D ±0.5% zone",
    )

    ax.axvline(
        event_time,
        linestyle="--",
        linewidth=1.3,
        label="Selected touch",
    )

    touch_row = window[
        window["open_time"].eq(event_time)
    ]

    if not touch_row.empty:
        ax.scatter(
            touch_row["open_time"],
            touch_row["close"],
            s=80,
            zorder=5,
            label="Touch candle",
        )

    ax.set_title(
        f"{ASSET} {TIMEFRAME} — 15D ±0.5% Touch — "
        f"{regime.title()} Regime\nTouch: {event_time}"
    )
    ax.set_xlabel("UTC time")
    ax.set_ylabel("Price")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.autofmt_xdate()
    fig.tight_layout()

    output = (
        FIGURE_DIRS["price_average"]
        / f"{ASSET}_{TIMEFRAME}_15d_zone_0_5_{regime}_example.png"
    )

    save_figure(fig, output)
    price_average_outputs.append(output)

print("Price/average charts:", len(price_average_outputs))


## 4. Forward-return distributions

In [ ]:

forward_statistics = []
forward_outputs = []

for horizon in HORIZONS:
    column = f"return_{horizon}_pct"

    values = pd.to_numeric(
        events[column],
        errors="coerce",
    ).dropna()

    median_return = values.median()
    mean_return = values.mean()

    p01 = values.quantile(0.01)
    p99 = values.quantile(0.99)

    plot_values = values[
        values.between(p01, p99)
    ]

    fig, ax = plt.subplots(figsize=(12, 7))

    ax.hist(
        plot_values,
        bins=80,
        alpha=0.8,
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1.2,
        label="0% return",
    )

    ax.axvline(
        median_return,
        linewidth=1.6,
        label=f"Median: {median_return:.3f}%",
    )

    ax.axvline(
        mean_return,
        linestyle=":",
        linewidth=1.6,
        label=f"Mean: {mean_return:.3f}%",
    )

    ax.set_title(
        f"{ASSET} {TIMEFRAME} — {AVERAGE_TYPE.upper()} ±{TOUCH_ZONE}% Touch\n"
        f"Forward Return Distribution — {horizon}"
    )
    ax.set_xlabel("Forward return (%)")
    ax.set_ylabel("Event count")
    ax.legend()
    ax.grid(alpha=0.25)

    fig.text(
        0.5,
        0.01,
        f"Valid events: {len(values):,} | Displayed range: 1st–99th percentile",
        ha="center",
    )

    fig.tight_layout(rect=[0, 0.04, 1, 1])

    output = (
        FIGURE_DIRS["forward_returns"]
        / f"{ASSET}_{TIMEFRAME}_{AVERAGE_TYPE}_zone_0_5_return_{horizon}_distribution.png"
    )

    save_figure(fig, output)
    forward_outputs.append(output)

    forward_statistics.append(
        {
            "horizon": horizon,
            "valid_count": len(values),
            "median_return_pct": median_return,
            "mean_return_pct": mean_return,
            "p01_return_pct": p01,
            "p99_return_pct": p99,
        }
    )

forward_statistics = pd.DataFrame(forward_statistics)

forward_statistics.to_csv(
    FIGURE_DIRS["forward_returns"]
    / "forward_return_distribution_statistics.csv",
    index=False,
)

display(forward_statistics)


## 5. MFE / MAE distributions

In [ ]:

mfe_mae_outputs = []

mae_values = events["mae_24h_pct"].dropna()
mfe_values = events["mfe_24h_pct"].dropna()

# MAE distribution
mae_p01 = mae_values.quantile(0.01)
mae_p99 = mae_values.quantile(0.99)
mae_plot = mae_values[mae_values.between(mae_p01, mae_p99)]

fig, ax = plt.subplots(figsize=(12, 7))
ax.hist(mae_plot, bins=80, alpha=0.8)
ax.axvline(
    mae_values.median(),
    linewidth=1.6,
    label=f"Median: {mae_values.median():.3f}%",
)
ax.axvline(
    mae_values.mean(),
    linestyle=":",
    linewidth=1.6,
    label=f"Mean: {mae_values.mean():.3f}%",
)
ax.axvline(0, linestyle="--", linewidth=1.2, label="0%")
ax.set_title(
    f"{ASSET} {TIMEFRAME} — {AVERAGE_TYPE.upper()} ±{TOUCH_ZONE}% Touch\n"
    "24h Maximum Adverse Excursion (MAE)"
)
ax.set_xlabel("MAE (%)")
ax.set_ylabel("Event count")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()

output = (
    FIGURE_DIRS["mfe_mae"]
    / "SOLUSDT_5m_15d_zone_0_5_mae_24h_distribution.png"
)
save_figure(fig, output)
mfe_mae_outputs.append(output)

# MFE distribution
mfe_p01 = mfe_values.quantile(0.01)
mfe_p99 = mfe_values.quantile(0.99)
mfe_plot = mfe_values[mfe_values.between(mfe_p01, mfe_p99)]

fig, ax = plt.subplots(figsize=(12, 7))
ax.hist(mfe_plot, bins=80, alpha=0.8)
ax.axvline(
    mfe_values.median(),
    linewidth=1.6,
    label=f"Median: {mfe_values.median():.3f}%",
)
ax.axvline(
    mfe_values.mean(),
    linestyle=":",
    linewidth=1.6,
    label=f"Mean: {mfe_values.mean():.3f}%",
)
ax.axvline(0, linestyle="--", linewidth=1.2, label="0%")
ax.set_title(
    f"{ASSET} {TIMEFRAME} — {AVERAGE_TYPE.upper()} ±{TOUCH_ZONE}% Touch\n"
    "24h Maximum Favorable Excursion (MFE)"
)
ax.set_xlabel("MFE (%)")
ax.set_ylabel("Event count")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()

output = (
    FIGURE_DIRS["mfe_mae"]
    / "SOLUSDT_5m_15d_zone_0_5_mfe_24h_distribution.png"
)
save_figure(fig, output)
mfe_mae_outputs.append(output)

# Scatter
complete = events[
    ["mae_24h_pct", "mfe_24h_pct"]
].dropna().copy()

complete["absolute_mae_pct"] = complete["mae_24h_pct"].abs()

mae_limit = complete["absolute_mae_pct"].quantile(0.99)
mfe_limit = complete["mfe_24h_pct"].quantile(0.99)

scatter_data = complete[
    (complete["absolute_mae_pct"] <= mae_limit)
    & (complete["mfe_24h_pct"] <= mfe_limit)
]

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(
    scatter_data["absolute_mae_pct"],
    scatter_data["mfe_24h_pct"],
    alpha=0.18,
    s=10,
)

diagonal_limit = max(
    scatter_data["absolute_mae_pct"].max(),
    scatter_data["mfe_24h_pct"].max(),
)

ax.plot(
    [0, diagonal_limit],
    [0, diagonal_limit],
    linestyle="--",
    linewidth=1.2,
    label="MFE = |MAE|",
)

ax.set_title(
    f"{ASSET} {TIMEFRAME} — {AVERAGE_TYPE.upper()} ±{TOUCH_ZONE}% Touch\n"
    "24h MFE vs Absolute MAE"
)
ax.set_xlabel("Absolute MAE (%)")
ax.set_ylabel("MFE (%)")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()

output = (
    FIGURE_DIRS["mfe_mae"]
    / "SOLUSDT_5m_15d_zone_0_5_mfe_vs_mae_scatter.png"
)
save_figure(fig, output)
mfe_mae_outputs.append(output)

# Event-level ratio distribution
ratio_data = complete[
    complete["absolute_mae_pct"] > 0
].copy()

ratio_data["mfe_mae_ratio"] = (
    ratio_data["mfe_24h_pct"]
    / ratio_data["absolute_mae_pct"]
)

ratio_values = ratio_data[
    np.isfinite(ratio_data["mfe_mae_ratio"])
]["mfe_mae_ratio"]

ratio_p01 = ratio_values.quantile(0.01)
ratio_p99 = ratio_values.quantile(0.99)
ratio_plot = ratio_values[
    ratio_values.between(ratio_p01, ratio_p99)
]

fig, ax = plt.subplots(figsize=(12, 7))
ax.hist(ratio_plot, bins=80, alpha=0.8)
ax.axvline(1, linestyle="--", linewidth=1.2, label="MFE = |MAE|")
ax.axvline(
    ratio_values.median(),
    linewidth=1.6,
    label=f"Event-level median: {ratio_values.median():.3f}",
)
ax.set_title(
    f"{ASSET} {TIMEFRAME} — {AVERAGE_TYPE.upper()} ±{TOUCH_ZONE}% Touch\n"
    "Event-Level 24h MFE / |MAE| Distribution"
)
ax.set_xlabel("MFE / |MAE| ratio")
ax.set_ylabel("Event count")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()

output = (
    FIGURE_DIRS["mfe_mae"]
    / "SOLUSDT_5m_15d_zone_0_5_mfe_mae_ratio_distribution.png"
)
save_figure(fig, output)
mfe_mae_outputs.append(output)

mfe_mae_stats = pd.DataFrame(
    [
        {
            "metric": "mae_24h",
            "valid_count": len(mae_values),
            "median": mae_values.median(),
            "mean": mae_values.mean(),
            "p01": mae_p01,
            "p99": mae_p99,
        },
        {
            "metric": "mfe_24h",
            "valid_count": len(mfe_values),
            "median": mfe_values.median(),
            "mean": mfe_values.mean(),
            "p01": mfe_p01,
            "p99": mfe_p99,
        },
        {
            "metric": "event_level_mfe_mae_ratio",
            "valid_count": len(ratio_values),
            "median": ratio_values.median(),
            "mean": ratio_values.mean(),
            "p01": ratio_p01,
            "p99": ratio_p99,
        },
    ]
)

mfe_mae_stats.to_csv(
    FIGURE_DIRS["mfe_mae"]
    / "mfe_mae_distribution_statistics.csv",
    index=False,
)

display(mfe_mae_stats)


## 6. Average / timeframe / asset / touch-zone comparisons

In [ ]:

comparison_outputs = []

# ---------------------------------------------------------
# Average comparison
# ---------------------------------------------------------

avg = average_summary.copy()
avg["average_type"] = avg["average_type"].astype(str).str.lower()
avg["order"] = avg["average_type"].map({"7d": 0, "15d": 1, "30d": 2})
avg = avg.sort_values("order")

labels = avg["average_type"].str.upper().tolist()

comparison_outputs.append(
    save_bar_chart(
        labels,
        avg["reach_probability_pct"].astype(float).tolist(),
        "Average Comparison — +2.5% Reach Probability",
        "+2.5% reach probability (%)",
        FIGURE_DIRS["average"] / "average_reach_probability.png",
    )
)

comparison_outputs.append(
    save_bar_chart(
        labels,
        avg["median_mfe_mae_ratio"].astype(float).tolist(),
        "Average Comparison — Median MFE / MAE Balance",
        "Median MFE / |MAE| ratio",
        FIGURE_DIRS["average"] / "average_mfe_mae_ratio.png",
        reference_line=1.0,
    )
)

# ---------------------------------------------------------
# Timeframe comparison
# ---------------------------------------------------------

tf = timeframe_summary.copy()
tf["order"] = tf["timeframe"].map({"1m": 0, "3m": 1, "5m": 2})
tf = tf.sort_values("order")

labels = tf["timeframe"].tolist()

comparison_outputs.append(
    save_bar_chart(
        labels,
        tf["reach_probability_pct"].astype(float).tolist(),
        "Timeframe Comparison — +2.5% Reach Probability",
        "+2.5% reach probability (%)",
        FIGURE_DIRS["timeframe"] / "timeframe_reach_probability.png",
    )
)

comparison_outputs.append(
    save_bar_chart(
        labels,
        tf["event_count"].astype(float).tolist(),
        "Timeframe Comparison — Median Event Count",
        "Median event count",
        FIGURE_DIRS["timeframe"] / "timeframe_event_count.png",
        fmt="{:,.0f}",
    )
)

# ---------------------------------------------------------
# Asset comparison
# ---------------------------------------------------------

asset = asset_summary.copy()
asset["order"] = asset["asset"].map(
    {"BTCUSDT": 0, "ETHUSDT": 1, "SOLUSDT": 2}
)
asset = asset.sort_values("order")
labels = asset["asset"].str.replace("USDT", "", regex=False).tolist()

comparison_outputs.append(
    save_bar_chart(
        labels,
        asset["reach_probability_pct"].astype(float).tolist(),
        "Asset Comparison — +2.5% Reach Probability",
        "+2.5% reach probability (%)",
        FIGURE_DIRS["asset"] / "asset_reach_probability.png",
    )
)

comparison_outputs.append(
    save_bar_chart(
        labels,
        asset["mfe_24h_median_pct"].astype(float).tolist(),
        "Asset Comparison — Median 24h Maximum Favorable Excursion",
        "Median MFE (%)",
        FIGURE_DIRS["asset"] / "asset_median_mfe_24h.png",
    )
)

comparison_outputs.append(
    save_bar_chart(
        labels,
        asset["mae_24h_median_pct"].astype(float).tolist(),
        "Asset Comparison — Median 24h Maximum Adverse Excursion",
        "Median MAE (%)",
        FIGURE_DIRS["asset"] / "asset_median_mae_24h.png",
        reference_line=0,
    )
)

# ---------------------------------------------------------
# Touch-zone comparison for primary SOL / 5m / 15d setup
# ---------------------------------------------------------

master = normalize_setup_columns(summary)

zone = master[
    (master["asset"] == ASSET)
    & (master["timeframe"] == TIMEFRAME)
    & (master["average_type"] == AVERAGE_TYPE)
].copy()

zone = zone.sort_values("touch_zone")

labels = [
    f"±{value:.1f}%"
    for value in zone["touch_zone"]
]

comparison_outputs.append(
    save_bar_chart(
        labels,
        zone["median_mfe_mae_ratio"].astype(float).tolist(),
        "SOLUSDT 5m / 15D — Touch-Zone MFE / MAE Comparison",
        "Median MFE / |MAE| ratio",
        FIGURE_DIRS["zone"] / "primary_setup_touch_zone_mfe_mae_ratio.png",
        reference_line=1.0,
    )
)

print("Comparison charts:", len(comparison_outputs))


## 7. Market-regime comparison

In [ ]:

regime_outputs = []

regime = regime_primary.copy()
regime["market_regime"] = regime["market_regime"].astype(str).str.lower()

order = ["bearish", "neutral", "bullish"]
regime["order"] = regime["market_regime"].map(
    {value: index for index, value in enumerate(order)}
)
regime = regime.sort_values("order")

labels = regime["market_regime"].str.title().tolist()

regime_metrics = [
    (
        "reach_2_5_probability_pct",
        "+2.5% Reach Probability",
        "+2.5% reach probability (%)",
        "regime_reach_probability.png",
        None,
    ),
    (
        "mae_24h_median_pct",
        "Median 24h MAE",
        "Median MAE (%)",
        "regime_median_mae.png",
        0,
    ),
    (
        "mfe_24h_median_pct",
        "Median 24h MFE",
        "Median MFE (%)",
        "regime_median_mfe.png",
        None,
    ),
    (
        "median_mfe_mae_ratio",
        "Median MFE / MAE Balance",
        "Median MFE / |MAE| ratio",
        "regime_mfe_mae_ratio.png",
        1.0,
    ),
    (
        "target_first_decisive_pct",
        "Target-First Decisive Rate",
        "Target-first decisive (%)",
        "regime_target_first_decisive.png",
        None,
    ),
    (
        "post_activation_median_extra_upside_24h_pct",
        "Post-Activation Extra Upside",
        "Median extra upside (%)",
        "regime_post_activation_upside.png",
        None,
    ),
]

for column, title, ylabel, filename, reference in regime_metrics:
    if column not in regime.columns:
        print("Skipping missing regime column:", column)
        continue

    regime_outputs.append(
        save_bar_chart(
            labels,
            regime[column].astype(float).tolist(),
            f"Primary Setup Regime Comparison — {title}",
            ylabel,
            FIGURE_DIRS["regime"] / filename,
            reference_line=reference,
        )
    )

print("Regime charts:", len(regime_outputs))


## 8. Profit-target reach and target-vs-stop first-hit

In [ ]:

target_outputs = []

# ---------------------------------------------------------
# Profit target reach curve
# ---------------------------------------------------------

reach = filter_primary_setup(profit_reach)

target_column = first_existing_column(
    reach,
    [
        "profit_target_pct",
        "target_pct",
        "profit_level_pct",
        "target_level_pct",
        "profit_target",
    ],
)

reach_probability_column = first_existing_column(
    reach,
    [
        "reach_probability_pct",
        "probability_pct",
        "target_reach_probability_pct",
        "reach_pct",
    ],
)

reach[target_column] = pd.to_numeric(
    reach[target_column],
    errors="coerce",
)

reach = reach.sort_values(target_column)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    reach[target_column],
    reach[reach_probability_column],
    marker="o",
)

ax.set_title(
    "SOLUSDT 5m / 15D / ±0.5% — Profit Target Reach"
)
ax.set_xlabel("Profit target (%)")
ax.set_ylabel("Reach probability (%)")
ax.grid(alpha=0.25)
fig.tight_layout()

output = (
    FIGURE_DIRS["target_first_hit"]
    / "primary_setup_profit_target_reach_curve.png"
)

save_figure(fig, output)
target_outputs.append(output)

# ---------------------------------------------------------
# Target vs stop first-hit
# ---------------------------------------------------------

fh = filter_primary_setup(first_hit)

stop_column = first_existing_column(
    fh,
    [
        "stop_pct",
        "stop_level_pct",
        "stop_loss_pct",
        "stop_threshold_pct",
        "stop_level",
    ],
)

decisive_column = first_existing_column(
    fh,
    [
        "target_first_decisive_pct",
        "decisive_target_first_pct",
        "target_first_pct_decisive",
    ],
)

all_confirmed_column = first_existing_column(
    fh,
    [
        "target_first_all_confirmed_pct",
        "target_first_pct",
        "target_first_all_pct",
    ],
    required=False,
)

fh[stop_column] = pd.to_numeric(
    fh[stop_column],
    errors="coerce",
)

fh = fh.sort_values(stop_column)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    fh[stop_column],
    fh[decisive_column],
    marker="o",
    label="Decisive target-first %",
)

if all_confirmed_column is not None:
    ax.plot(
        fh[stop_column],
        fh[all_confirmed_column],
        marker="o",
        label="All-confirmed target-first %",
    )

ax.axhline(
    50,
    linestyle="--",
    linewidth=1.2,
    label="50%",
)

ax.set_title(
    "SOLUSDT 5m / 15D / ±0.5% — +2.5% Target vs Stop First-Hit"
)
ax.set_xlabel("Stop level (%)")
ax.set_ylabel("Target-first rate (%)")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()

output = (
    FIGURE_DIRS["target_first_hit"]
    / "primary_setup_target_vs_stop_first_hit.png"
)

save_figure(fig, output)
target_outputs.append(output)

print("Target / first-hit charts:", len(target_outputs))


## 9. Trailing and post-activation behavior

In [ ]:

trailing_outputs = []

trail = filter_primary_setup(trailing)

giveback_column = first_existing_column(
    trail,
    [
        "giveback_pct",
        "trail_giveback_pct",
        "profit_giveback_pct",
        "trailing_giveback_pct",
        "giveback",
    ],
    required=False,
)

median_return_column = first_existing_column(
    trail,
    [
        "median_realized_return_pct",
        "trail_median_realized_return_pct",
        "median_locked_profit_pct",
        "trail30_median_realized_return_pct",
    ],
    required=False,
)

resolved_column = first_existing_column(
    trail,
    [
        "resolved_pct",
        "trail_resolved_pct",
        "resolved_rate_pct",
    ],
    required=False,
)

ambiguous_column = first_existing_column(
    trail,
    [
        "ambiguous_pct",
        "trail_ambiguous_pct",
        "same_candle_ambiguous_pct",
    ],
    required=False,
)

no_trigger_column = first_existing_column(
    trail,
    [
        "no_trigger_pct",
        "trail_no_trigger_pct",
        "no_trigger_rate_pct",
    ],
    required=False,
)

# ---------------------------------------------------------
# Giveback vs median realized return
# ---------------------------------------------------------

if giveback_column and median_return_column:
    trail[giveback_column] = pd.to_numeric(
        trail[giveback_column],
        errors="coerce",
    )

    trail = trail.sort_values(giveback_column)

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(
        trail[giveback_column],
        trail[median_return_column],
        marker="o",
    )

    ax.set_title(
        "SOLUSDT 5m / 15D / ±0.5% — Trailing Giveback vs Median Return"
    )
    ax.set_xlabel("Profit giveback (%)")
    ax.set_ylabel("Median realized return (%)")
    ax.grid(alpha=0.25)
    fig.tight_layout()

    output = (
        FIGURE_DIRS["trailing"]
        / "primary_setup_trailing_giveback_vs_return.png"
    )

    save_figure(fig, output)
    trailing_outputs.append(output)
else:
    print(
        "Skipping giveback-vs-return chart because expected columns "
        "were not found."
    )

# ---------------------------------------------------------
# Resolved / ambiguous / no-trigger comparison
# ---------------------------------------------------------

if (
    giveback_column
    and resolved_column
    and ambiguous_column
    and no_trigger_column
):
    fig, ax = plt.subplots(figsize=(11, 6))

    x = np.arange(len(trail))
    width = 0.25

    ax.bar(
        x - width,
        trail[resolved_column],
        width,
        label="Resolved",
    )

    ax.bar(
        x,
        trail[ambiguous_column],
        width,
        label="Ambiguous",
    )

    ax.bar(
        x + width,
        trail[no_trigger_column],
        width,
        label="No trigger",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [
            f"{value:g}%"
            for value in trail[giveback_column]
        ]
    )

    ax.set_title(
        "SOLUSDT 5m / 15D / ±0.5% — Trailing Outcome Quality"
    )
    ax.set_xlabel("Profit giveback (%)")
    ax.set_ylabel("Outcome share (%)")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()

    output = (
        FIGURE_DIRS["trailing"]
        / "primary_setup_trailing_outcome_quality.png"
    )

    save_figure(fig, output)
    trailing_outputs.append(output)
else:
    print(
        "Skipping trailing quality chart because expected columns "
        "were not found."
    )

# ---------------------------------------------------------
# Primary 30% giveback outcome summary from Stage 1 master
# ---------------------------------------------------------

primary_master = filter_primary_setup(summary)

if len(primary_master) == 1:
    primary_master = primary_master.iloc[0]

    candidates = {
        "Resolved": "trail30_resolved_pct",
        "Ambiguous": "trail30_ambiguous_pct",
        "No trigger": "trail30_no_trigger_pct",
    }

    if all(column in primary_master.index for column in candidates.values()):
        labels = list(candidates.keys())
        values = [
            float(primary_master[column])
            for column in candidates.values()
        ]

        trailing_outputs.append(
            save_bar_chart(
                labels,
                values,
                "Primary Setup — 30% Giveback Outcome Quality",
                "Outcome share (%)",
                FIGURE_DIRS["trailing"]
                / "primary_setup_30pct_giveback_quality.png",
            )
        )

print("Trailing charts:", len(trailing_outputs))


## 10. Final visualization index and validation

In [ ]:

all_pngs = sorted(
    FIGURE_ROOT.rglob("*.png")
)

index_rows = []

for path in all_pngs:
    index_rows.append(
        {
            "file": str(path),
            "category": path.parent.name,
            "size_bytes": path.stat().st_size,
            "non_empty": path.stat().st_size > 0,
        }
    )

visualization_index = pd.DataFrame(index_rows)

index_file = (
    FIGURE_ROOT
    / "stage1_visualization_index.csv"
)

visualization_index.to_csv(
    index_file,
    index=False,
)

print("Total PNG charts:", len(visualization_index))
print("All non-empty:", bool(visualization_index["non_empty"].all()))
print("Index:", index_file)

display(
    visualization_index.groupby("category")
    .size()
    .rename("chart_count")
    .reset_index()
)


In [ ]:

# ---------------------------------------------------------
# Validate forward-return chart statistics against the final
# Stage 1 configuration summary.
# ---------------------------------------------------------

primary_summary = filter_primary_setup(summary)

if len(primary_summary) != 1:
    raise ValueError(
        f"Expected exactly one primary summary row, found {len(primary_summary)}"
    )

primary_summary = primary_summary.iloc[0]

validation_rows = []

for _, row in forward_statistics.iterrows():
    horizon = row["horizon"]

    count_ok = (
        int(row["valid_count"])
        == int(primary_summary[f"return_{horizon}_valid_count"])
    )

    median_ok = np.isclose(
        row["median_return_pct"],
        primary_summary[f"return_{horizon}_median_pct"],
        atol=1e-12,
    )

    mean_ok = np.isclose(
        row["mean_return_pct"],
        primary_summary[f"return_{horizon}_mean_pct"],
        atol=1e-12,
    )

    validation_rows.append(
        {
            "horizon": horizon,
            "count_ok": count_ok,
            "median_ok": median_ok,
            "mean_ok": mean_ok,
            "pass": count_ok and median_ok and mean_ok,
        }
    )

validation = pd.DataFrame(validation_rows)

validation_file = (
    FIGURE_ROOT
    / "stage1_visualization_validation.csv"
)

validation.to_csv(
    validation_file,
    index=False,
)

display(validation)

print(
    "\nForward-return validation:",
    f"{validation['pass'].sum()}/{len(validation)} passed",
)

if not validation["pass"].all():
    raise ValueError(
        "Forward-return visualization validation failed."
    )

if not visualization_index["non_empty"].all():
    raise ValueError(
        "One or more visualization files are empty."
    )

print("\nPART 23 VISUALIZATION NOTEBOOK COMPLETED SUCCESSFULLY")



## Interpretation guardrails

Use the charts to **support** Stage 1 findings, not to convert them into a finalized strategy.

Current Stage 1 interpretation remains:

- `15D` = strongly supported primary average
- `SOLUSDT` = strongest upside/opportunity candidate, not universally safest
- `1m ≈ 3m ≈ 5m` behaviorally; `5m` is mainly an operational candidate
- `±0.5%` = leading touch-zone candidate, not final threshold
- `Neutral` = strongest balanced regime
- `Bearish` = stronger rebound opportunity but higher risk
- `+2.5%` = activation candidate
- `30% profit giveback` = practical trailing candidate

These plots are descriptive research outputs. Stage 2 still needs trade-state rules, deduplication, fees, slippage, position logic, and out-of-sample testing.
